## **15. 특징 추출 및 변환**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# 데이터 읽어오기

def parser(x):
    return datetime.strptime(x, '%Y-%m-%d %H:%M:%S')

input_file = './data/AirQualityUCI_refined.csv'

df = pd.read_csv(input_file,
                 index_col=[0],
                 parse_dates=[0],
                 date_parser=parser)

df.head()

In [ ]:
# [+] 일산화탄소 변수 결측 데이터 처리
co = df['CO(GT)'].copy()  # 복사본 생성
co.interpolate(inplace=True) # 선형 보간 (inplace=True를 써야지 덮어쓰기가 됨.)

### **구간화**(binning)

In [ ]:
# [+] 최대값, 최소값
max_val = co.max()
min_val = co.min()

print(max_val, min_val)

In [ ]:
# [+] 구간 별 기준 값 집합 생성
bins = np.linspace(min_val, max_val, 6)
bins

In [ ]:
# 구간에 대한 레이블 집합
labels=['0 <=x<2.38', '2.38<=x<4.76', '4.76<=x<7.14',
       '7.14<=x<9.52', '9.52<=x<11.9']

In [ ]:
# 일산화탄소 변수(수치형)에 대한 범주형 변수 생성
df['bins'] = pd.cut(
    co, 
    bins=bins, 
    labels=labels, 
    include_lowest=True # 첫 번째 구간의 왼쪽 경계값을 포함할지 여부
)

df.info()

In [ ]:
# bins 변수 출력
df['bins']

In [ ]:
# bins 변수의 히스토그램 시각화
plt.hist(sorted(df['bins']), bins=len(bins)-1)
plt.show()

### **로그 변환**(log transformation)

In [ ]:
# 질소 산화물 분포 시각화
sns.displot(df['PT08.S3(NOx)'], kde=True)

In [ ]:
# [+] 로그 스케일로 변환: Common logarithm(log10)
df['log'] = np.log10(df['PT08.S3(NOx)'])

In [ ]:
# 변환된 변수 출력
df['log']

In [ ]:
# 변환된 변수의 분포 시각화
sns.displot(df['log'], kde=True)
plt.xlabel('log(NOx)')
plt.show()

### **원핫인코딩**(one-hot encoding)

In [ ]:
# 예제 데이터: 인사 평가

emp_id = pd.Series([1, 2, 3, 4, 5])
gender = pd.Series(['Male', 'Female', 'Female', 'Male', 'Female'])
remarks = pd.Series(['Nice', 'Good', 'Great', 'Great', 'Nice'])

df_emp = pd.DataFrame()
df_emp['emp_id'] = emp_id
df_emp['gender'] = gender
df_emp['remarks'] = remarks

df_emp

In [ ]:
# [+] 범주형 변수 별(gender, remarks) Unique value 리스트 출력
print(df_emp['gender'].unique()) 
print(df_emp['remarks'].unique())

In [ ]:
# [+] 원핫인코딩 적용: 범주형 변수 -> 이진값 벡터
df_emp_encoded = pd.get_dummies(
    df_emp,
    columns=['gender', 'remarks'],
    dtype=float
)
df_emp_encoded

### **정규화**(normalization)

In [ ]:
# [+] 비메탄탄화수소 변수 결측 데이터 처리
nmhc = df['PT08.S2(NMHC)'].copy() # 복사본 생성
nmhc.interpolate(inplace=True) # 결측치 처리

In [ ]:
# 스케일이 서로 다른 두 변수 시각화
plt.plot(co, label='Carbon monooxide')
plt.plot(nmhc, label='Non-methane hydrocarbon')
plt.ylabel('Concentration')
plt.legend(loc='best')

In [ ]:
# [+] 일산화탄소 변수 정규화
co_max = co.max()  # 최대값
co_min = co.min()  # 최소값

# 최소-최대 정규화
df['CO_Norm'] = (co - co_min) / (co_max - co_min)
df['CO_Norm']

In [ ]:
# [+] 비메탄탄화수소 변수 정규화
nmhc_max = nmhc.max()
nmhc_min = nmhc.min()

df['NMHC_Norm'] = (nmhc - nmhc_min) / (nmhc_max - nmhc_min) 
df['NMHC_Norm']

In [ ]:
# 정규화된 두 변수 시각화
plt.plot(df['CO_Norm'], label='Carbon monooxide')
plt.plot(df['NMHC_Norm'], label='Non-methane hydrocarbon')
plt.ylabel('Concentration')
plt.legend(loc='best')

### **특징 분할**

In [ ]:
# 예제: 영화 데이터
movies = pd.Series(["The Godfather, 1972, Francis Ford Coppola",
                    "Contact, 1997, Robert Zemeckis",
                   "Parasite, 2019, Joon-ho Bong"])

movies

In [ ]:
# 영화 데이터의 값 부
# Divide movie data into title, year, director columns
lst_title = []
lst_year = []
lst_director = []

for val in movies:
    title, year, director = val.split(',')  # data split
    lst_title.append(title)
    lst_year.append(year)
    lst_director.append(director)

print(lst_title)
print(lst_year)
print(lst_director)

In [ ]:
# Make a DataFrame object
df_movie = pd.DataFrame()
df_movie['title'] = lst_title
df_movie['year'] = lst_year
df_movie['director'] = lst_director

df_movie